In [1]:
import pandas as pd
import numpy as np
from Levenshtein import distance as levenshtein_distance

hisco_df = pd.read_excel("../../../data/H4/normalisering/HISCO-HISCLASS_nl-fr.xlsx")
machinaal_df = pd.read_excel("../../../data/H4/normalisering/machinaal_test.xlsx")

def preprocess(text):
    if isinstance(text, str):
        return text.lower().strip()
    return np.nan

hisco_df['profession_clean'] = hisco_df['profession'].apply(preprocess)
machinaal_df['P_PROF_clean'] = machinaal_df['P_PROF'].apply(preprocess)
hisco_prof_list = hisco_df['profession_clean'].dropna().unique().tolist()

def match_levenshtein(prof, choices, max_distance_ratio=0.3):
    if pd.isna(prof):
        return np.nan, np.nan
    best_match = None
    best_score = float('inf')
    best_raw_distance = np.nan  # Default to NaN
    for choice in choices:
        dist = levenshtein_distance(prof, choice)
        norm_dist = dist / max(len(prof), len(choice))
        if norm_dist < best_score and norm_dist <= max_distance_ratio:
            best_score = norm_dist
            best_match = choice
            best_raw_distance = dist
    if best_match is None:
        best_match = 'unknown'
    return best_match, best_raw_distance

unique_profs = machinaal_df['P_PROF_clean'].dropna().unique()
match_results = {prof: match_levenshtein(prof, hisco_prof_list, 0.3) for prof in unique_profs}

match_dict = {prof: results[0] for prof, results in match_results.items()}
distance_dict = {prof: results[1] for prof, results in match_results.items()}

machinaal_df['P_PROF_match'] = machinaal_df['P_PROF_clean'].map(match_dict)
machinaal_df['levenshtein_distance'] = machinaal_df['P_PROF_clean'].map(distance_dict)

hisco_reduced = hisco_df[['profession_clean', 'hisco', 'hisclass']].drop_duplicates(subset=['profession_clean'])

unknown_row = pd.DataFrame({
    'profession_clean': ['unknown'],
    'hisco': ['unknown'],
    'hisclass': ['unknown']
})
hisco_reduced = pd.concat([hisco_reduced, unknown_row], ignore_index=True)

# Perform the merge
merged_df = machinaal_df.merge(
    hisco_reduced,
    left_on='P_PROF_match',
    right_on='profession_clean',
    how='left'
)

duplicate_check = merged_df.duplicated(subset=['Person ID', 'archive_file'], keep=False)
if duplicate_check.any():
    print(f"Found {duplicate_check.sum()} duplicate rows based on Person ID and archive_file")
    
    # Remove duplicates, keeping the first occurrence
    merged_df = merged_df.drop_duplicates(subset=['Person ID', 'archive_file'], keep='first')
    print(f"Removed duplicates. Final dataset has {len(merged_df)} rows")

merged_df = merged_df.rename(columns={
    'hisco': 'P_PROF_hisco',
    'hisclass': 'P_PROF_hisclass'
})

merged_df.drop(columns=['P_PROF_clean', 'profession_clean', 'levenshtein_distance'], inplace=True)

merged_df.to_excel("../../../data/H4/normalisering/machinaal_test_prof.xlsx", index=False)
print(f"Saved file with {len(merged_df)} unique records")

Found 2971 duplicate rows based on Person ID and archive_file
Removed duplicates. Final dataset has 12203 rows


Saved file with 12203 unique records
